# Session 3 - Feature Engineering, Pipelines & Data Leakage

**Block 1: Data Science Fundamentals** · 4 hours

---

## Learning objectives

By the end of this session you will be able to:

1. Build a preprocessing pipeline in which nothing is fitted on data it should not
   see. `[CLO3]`
2. Identify at least three distinct leakage mechanisms in an analysis, and
   **quantify** the inflation each one causes. `[CLO3]`
3. Predict, before running, whether scaling will change a given model's
   performance - and explain why. `[CLO6]`
4. Engineer and defend features on domain grounds. `[CLO1]`
5. Explain why target encoding leaks, and how to use it safely. `[CLO3]`

## Prerequisites

Sessions 1–2. You need your cleaning function and your sealed test set. If your
`data/SEALED_TEST/test.parquet` does not exist, stop and tell me now.

## Why does this matter?

Today you will build a model that performs beautifully, and then discover that
every good thing about it was an illusion. You will do this three times, for three
different reasons.

This is not a trick played on you for entertainment. Reviewing published machine
learning across seventeen scientific fields, Kapoor & Narayanan (2023) found
leakage-driven errors in **hundreds** of peer-reviewed papers. The people who made
those mistakes were not careless; they were domain experts who did not know what to
look for. Today is about learning what to look for.

## §1 - Retrieval practice

From memory, five minutes.

1. Why must the split come before cleaning?
2. A random split contaminated what percentage of our test set, and by what mechanism?
3. `review_scores_rating` is missing for which listings, exactly?
4. We dropped the price-missing rows. How did that change the population?
5. Name one thing a boxplot of price by room type cannot tell you.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data import SEED, load_raw, set_seed, split_by_host

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 150)
set_seed(SEED)

# Reproduce Session 2's split and cleaning. Training data only, always.
df = load_raw()
# The underscore discards the test half on purpose. It is sealed, and a variable we
# do not bind is a variable we cannot accidentally use.
train, _ = split_by_host(df)
train["price_num"] = (
    train["price"].astype(str).str.replace(r"[^0-9.]", "", regex=True)
    .replace("", np.nan).astype(float)
)
# Two scoping decisions, both made here and both worth naming aloud: we model only
# rows that have a price, and only prices in a plausible range. Each one changes the
# population the model describes, so each belongs in the decision log.
work = train[train["price_num"].notna()].copy()
work = work[work["price_num"].between(10, 2000)]
# log1p because price is multiplicative and right-skewed, as Session 2 showed.
y = np.log1p(work["price_num"])
# Carried alongside y so every cross-validation below can group by host.
groups = work["host_id"]

print(f"training rows        : {len(train):,}")
print(f"modelling rows       : {len(work):,}")
print(f"target: log1p(price), skew = {y.skew():.3f}")

## §2 - From insight to feature

Feature engineering is not "make more columns". It is **encoding a belief about the
world in a form the model can use**. Every feature should come with a sentence
explaining why you expect it to matter.

The corollary, which most courses skip: a feature that encodes a *correct* belief
can still fail to help, and you must be willing to measure that and throw it away.

### The amenities field

In [ ]:
# amenities arrives as a JSON array inside a text column, so it has to be parsed
# before it is anything. The isinstance guard keeps a missing value from crashing
# json.loads and returns an empty list instead.
amenities = work["amenities"].map(lambda s: json.loads(s) if isinstance(s, str) else [])
# A nested comprehension flattens the lists into one stream, and Counter tallies it.
counts = Counter(a for lst in amenities for a in lst)

print(f"distinct amenities        : {len(counts):,}")
print(f"mean amenities per listing: {amenities.map(len).mean():.1f}")
# The two lines below are the reason raw amenities cannot be one-hot encoded as is.
# Anything appearing once cannot generalise, and anything in most listings cannot
# discriminate. The useful signal is in the middle of that range.
print(f"appearing exactly once    : {sum(1 for v in counts.values() if v == 1):,}")
print(f"appearing in >50% listings: {sum(1 for v in counts.values() if v > len(work) * 0.5)}")
print("\nmost common:")
for name, n in counts.most_common(8):
    print(f"  {n:6,}  {name}")

Read those numbers as a design problem. 2,163 distinct amenities, of which 1,232
appear exactly once, and about twenty appear in over half of all listings.

- The ~20 near-universal ones carry almost no information: if everyone has Wifi,
  Wifi does not distinguish anyone.
- The 1,232 singletons carry no *generalisable* information: a feature that is true
  for one training row will describe nothing in the test set.
- The useful signal is in the middle band.

This shape - a few near-universal, a long tail of near-unique, signal in the
middle - is extremely common in real categorical data.

In [ ]:
# TODO: Build three features from `amenities` and justify each in a comment.
#
#   1. n_amenities        - how many amenities the listing lists
#   2. has_<something>    - a binary flag for one amenity you believe affects price.
#                           State your belief BEFORE you look at any correlation.
#   3. a feature of your own design
#
# Then check each against the target. Were you right?

### An intuitive feature that does not work

Everyone's first instinct for property pricing is distance to the centre. Let us
encode that belief properly and then test it honestly.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

CATALUNYA = (41.3874, 2.1700)  # Plaça de Catalunya


def haversine_km(lat, lon, point):
    # Great-circle distance on a sphere. Latitude and longitude are angles, so they
    # must be converted to radians before any trigonometry.
    lat1, lon1 = np.radians(lat), np.radians(lon)
    lat2, lon2 = np.radians(point[0]), np.radians(point[1])
    # The haversine formula. Using it rather than straight Pythagoras on degrees
    # matters because a degree of longitude is shorter than a degree of latitude,
    # and by a different amount at every latitude.
    h = (np.sin((lat1 - lat2) / 2) ** 2
         + np.cos(lat2) * np.cos(lat1) * np.sin((lon1 - lon2) / 2) ** 2)
    # 6371 km is the Earth's mean radius, so the result comes out in kilometres.
    return 6371 * 2 * np.arcsin(np.sqrt(h))


# Domain knowledge turned into a number. The model has latitude and longitude, but
# it has no idea that the middle of the city is expensive; this feature says so.
work["km_centre"] = haversine_km(work["latitude"], work["longitude"], CATALUNYA)
print(f"km to Plaça de Catalunya: median {work['km_centre'].median():.2f}, "
      f"max {work['km_centre'].max():.2f}")
# Negative as expected: further from the centre, lower price.
print(f"corr(log price, km_centre) = {np.corrcoef(y, work['km_centre'])[0, 1]:+.3f}")

# Group-aware cross-validation, from here to the end of the notebook. Folds are
# built so that a host's listings never straddle a fold boundary.
gkf = GroupKFold(n_splits=5)
base = ["accommodates", "bedrooms", "bathrooms", "latitude", "longitude"]


def knn_score(cols):
    # KNN is the right probe for a geographic feature: it works on distances, so it
    # is the model most likely to notice one. Scaling is inside the pipeline, which
    # is what makes the fold boundary honest.
    pipe = Pipeline([("i", SimpleImputer(strategy="median")),
                     ("s", StandardScaler()),
                     ("m", KNeighborsRegressor(n_neighbors=10))])
    return cross_val_score(pipe, work[cols], y, cv=gkf, groups=groups,
                           scoring="r2").mean()


# The controlled comparison: identical model, identical folds, one feature added.
# That is the only way the difference can be attributed to the feature.
without = knn_score(base)
with_km = knn_score(base + ["km_centre"])
print(f"\nKNN without km_centre : R² = {without:.4f}")
print(f"KNN with    km_centre : R² = {with_km:.4f}")
# A small change is a real answer. Latitude and longitude already carry most of
# this information, and a feature that restates what the model has is not free.
print(f"change                : {with_km - without:+.4f}")

It makes the model **worse**.

The belief was not wrong - central listings *are* more expensive, weakly
(r = −0.14). The feature is redundant: latitude and longitude already locate every
listing, and a distance-to-one-point summary throws away direction while adding a
dimension for the model to get lost in.

Keep this result. In Session 7 you will meet people who add features until the
score stops improving; this is why that is not a method.

## §3 - Encoding and scaling: predict, then run

### Categorical encoding, and the cardinality problem

We have three categorical columns at very different cardinalities, and they need
different treatment.

In [ ]:
# Cardinality drives the encoding decision, so measure it before choosing one.
# Four categorical columns, four very different problems.
for col in ["room_type", "neighbourhood_group_cleansed",
            "neighbourhood_cleansed", "property_type"]:
    vc = work[col].value_counts()
    # The last column is the one that matters: categories with fewer than 20
    # observations cannot be estimated reliably and will overfit if one-hot encoded
    # individually. This is the argument for min_frequency later.
    print(f"{col:32s} {vc.size:3d} categories   "
          f"rarest {vc.min():4d}   below 20 obs: {(vc < 20).sum():3d}")

One-hot encoding `neighbourhood_cleansed` produces 69 columns, some backed by a
handful of listings. A category with four observations cannot support a reliable
coefficient, and it will not appear at all in some cross-validation folds.

`OneHotEncoder(min_frequency=20)` groups everything rarer than 20 observations
into a single `infrequent` column. That is a *modelling* choice with a threshold
you should be able to defend.

**Target encoding** - replacing each category with the mean target value in that
category - is the tempting alternative. It produces one column instead of 69 and
usually improves cross-validation scores immediately.

It is also a trap, and we will return to it in §6. For now, note the shape of the
problem: to compute the mean price per neighbourhood, you must **look at the
target**. Anything that looks at the target during preprocessing needs very careful
handling.

### Scaling: does it matter?

The received wisdom is "always scale your features". Let us test it.

### Predict before you run

Two models - K-nearest neighbours and a decision tree - and two feature sets:

- **Set A:** `accommodates`, `bedrooms`, `latitude`, `longitude`
  (all on comparable, small scales)
- **Set B:** Set A plus `number_of_reviews`, `availability_365`, `maximum_nights`,
  `minimum_nights` (standard deviations from 1.2 up to 413)

For each of the four combinations, predict whether `StandardScaler` will help a
lot, help a little, do nothing, or hurt.

In [ ]:
# TODO: Fill in your four predictions before running the next cell.
#
#   Set A + KNN  :
#   Set A + Tree :
#   Set B + KNN  :
#   Set B + Tree :

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# A 2x2 experiment: two feature sets crossed with two models. Set B adds columns
# whose numeric range is enormous compared with the others, which is the condition
# that makes scaling matter.
SETS = {
    "A capacity + geo": ["accommodates", "bedrooms", "latitude", "longitude"],
    "B + large-scale ": ["accommodates", "bedrooms", "latitude", "longitude",
                         "number_of_reviews", "availability_365",
                         "maximum_nights", "minimum_nights"],
}
# KNN measures distances, so every feature competes on its raw scale.
# A tree splits on thresholds one feature at a time, so scale cannot reach it.
MODELS = {
    "KNN(k=10)": KNeighborsRegressor(n_neighbors=10),
    "Tree(d=8) ": DecisionTreeRegressor(max_depth=8, random_state=SEED),
}

# Print the spreads first. maximum_nights dwarfs everything else, and that single
# number predicts the whole table below.
print("feature standard deviations (Set B):")
print(work[SETS["B + large-scale "]].std().round(2).to_string())
print()
print(f"{'features':18s} {'model':11s} {'unscaled':>9s} {'scaled':>9s} {'change':>8s}")
for set_name, cols in SETS.items():
    for model_name, model in MODELS.items():
        # Identical except for the StandardScaler step, so the change column
        # isolates the effect of scaling and nothing else.
        raw = cross_val_score(
            Pipeline([("i", SimpleImputer(strategy="median")), ("m", model)]),
            work[cols], y, cv=gkf, groups=groups, scoring="r2").mean()
        scaled = cross_val_score(
            Pipeline([("i", SimpleImputer(strategy="median")),
                      ("s", StandardScaler()), ("m", model)]),
            work[cols], y, cv=gkf, groups=groups, scoring="r2").mean()
        print(f"{set_name:18s} {model_name:11s} {raw:>9.4f} {scaled:>9.4f} "
              f"{scaled - raw:>+8.4f}")

Four results, four different lessons:

| Combination | Effect | Why |
|---|---|---|
| Set A + KNN | ≈ nothing (−0.005) | features already on comparable scales |
| Set A + Tree | ≈ nothing | trees split on thresholds; a monotone rescaling cannot change which split is chosen |
| **Set B + KNN** | **+0.24** | `maximum_nights` (sd 413) dominated every distance calculation; the other features were invisible |
| Set B + Tree | ≈ nothing (−0.0002) | still invariant, regardless of scale spread |

So the rule is not "always scale". It is:

> **Scale when the model measures distances or is fitted by gradient descent, and
> when your features are on different scales. Trees do not care.**

Note also that in Set A, scaling was very slightly *harmful* to KNN. Any operation
can hurt; "harmless by default" is not a category that exists in modelling.

## §4 - Leakage Lab, part 1

Client B wants a price model. You have a cleaned training set and a target. Build
the obvious thing: use every numeric column that is not the target.

In [ ]:
# The naive approach, done deliberately: take every numeric column and model it.
# DROP removes only the obvious non-features, identifiers and the target itself.
DROP = {"price", "price_num", "id", "host_id", "scrape_id", "host_profile_id",
        "km_centre"}
naive_features = [
    c for c in work.columns
    if pd.api.types.is_numeric_dtype(work[c])
    and c not in DROP
    and not work[c].isna().all()
]
print(f"using {len(naive_features)} numeric features")

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge


def evaluate(cols, label):
    """Score a feature set with two very different models."""
    # Two models on purpose. A leak that inflates a linear model and a tree model
    # equally is a property of the data, not an artifact of one algorithm.
    row = {"features": len(cols)}
    for name, model in [("Ridge", Ridge()),
                        ("HistGB", HistGradientBoostingRegressor(random_state=SEED))]:
        s = cross_val_score(
            Pipeline([("i", SimpleImputer(strategy="median")),
                      ("s", StandardScaler()), ("m", model)]),
            work[cols], y, cv=gkf, groups=groups, scoring="r2")
        # The spread across folds is reported next to the mean, and from Session 6
        # onward it is the number that decides whether a difference is real.
        row[name] = f"{s.mean():.4f} +/- {s.std():.4f}"
    print(f"  {label:34s} n={row['features']:3d}   "
          f"Ridge {row['Ridge']}   HistGB {row['HistGB']}")
    return row


print()
# Stop at this output. An R² this high on a messy scraped dataset is not a triumph,
# it is a symptom, and the rest of the session is the diagnosis.
evaluate(naive_features, "every numeric column")

### Stop.

**R² = 0.999.**

That means the model explains 99.9% of the variance in nightly price across
thousands of listings, using only numeric columns. Every professional in this field
would tell you the honest ceiling for this kind of problem is somewhere around
0.5–0.7.

Notice also that the two models disagree wildly: 0.855 for Ridge, 0.999 for the
gradient booster. When two reasonable models disagree by that much, the data is
telling you something.

### Your task: find out why

Do not read on. Investigate. Twenty minutes.

Some questions worth asking:
- Which features is the model actually relying on?
- Is any feature suspiciously well correlated with the target?
- Could any feature be a copy of the target under another name?
- What would each feature's value be at the moment the client needs a prediction?

In [ ]:
# TODO: Investigate. Find out why the model scores 0.999.
#
# Suggested starting point: correlate every feature with the target and sort.

`price_quote_price_per_night` is byte-for-byte identical to `price` on every row
where both exist. The model was not predicting price; it was reading it.

### Why did Ridge only get 0.855?

This is a genuinely interesting detail. If the feature *is* the target, why did the
linear model not score 1.000?

Because we are modelling `log1p(price)`, and the leaking feature is on the raw euro
scale. Ridge can only form a linear combination of its inputs, and `log(price)` is
not a linear function of `price`. It gets as close as a straight line can. The
gradient booster, being nonlinear, has no such difficulty and reads the answer
straight off.

> **A leak is not equally visible to every model.** A flexible model exploits it
> fully and produces an obviously impossible score. A rigid model exploits it
> partially and produces a *plausible-looking* score. The second case is far more
> dangerous, because nothing looks wrong.

### Remove it and try again.

In [ ]:
# Remove the duplicate of the target and re-score. Note that the number falls but
# stays implausibly high, which is how you know one leak is not the whole story.
step2 = [c for c in naive_features if c != "price_quote_price_per_night"]
evaluate(step2, "minus the identical column")

0.973.

Still impossible. We are not done.

## §5 - The leakage taxonomy

**Data leakage** is the use of information during model development that would not
be available at prediction time, or that comes from data the evaluation is supposed
to be blind to.

It produces an evaluation that is optimistic by an unknown amount - and *unknown*
is the operative word. A model that is 3% worse than you think is a minor problem.
The trouble is that leakage gives you no way to estimate the size of your own error.

### Four mechanisms

**1. Target leakage.** A feature contains the answer - directly, or recoverably.
Detection: correlations near 1; impossible scores; asking whether the value exists
at prediction time. *We just found one.*

**2. Train–test contamination.** Information crosses the split. Fitting a scaler or
an imputer on all the data before splitting; deciding an outlier threshold by
looking at everything. Detection: any `.fit()` that ran before the split. Prevention:
pipelines.

**3. Group leakage.** The same entity appears on both sides, so the model recalls
rather than generalises. *We fixed this in Session 2 - 77.9% of a random test set
was contaminated by shared hosts.*

**4. Temporal leakage.** The future informs a prediction about the past. Not our
main hazard with a single snapshot, but the dominant one in any time-series work.

### Three detection habits

1. **The deployment test.** For each feature, ask: *at the instant the client needs
   this prediction, does this value exist?* This single question catches most target
   leakage, and it needs no computation.
2. **Distrust good news.** A result better than the field's known ceiling is a bug
   report, not an achievement.
3. **Read the column names, then read the data.** The best leak we will find today
   is invisible to a correlation ranking.

### A warning about detection by correlation

You found the first leak by ranking correlations. That worked, and it will fail you
next. Hold that thought for exactly one section.

## §6 - Leakage Lab, part 2 (independent)

We are at **HistGB R² = 0.973** with the obvious culprit removed. Something else is
leaking.

This one is harder. The remaining leak is not a copy of the target, and it will not
stand out in a correlation ranking. It is spread across **more than one column**.

Thirty minutes. Work alone first, then compare.

Hints, in escalating order - use as few as you can:
1. The leak involves columns whose names describe *outcomes* rather than properties.
2. Two columns, neither alarming alone, become the target when combined.
3. Price is money per night. What else in this dataset is money, and what else is
   nights?

In [ ]:
# TODO: Find the remaining leak(s). For each one, report:
#   - which columns are involved
#   - the mechanism (how the target is recoverable)
#   - the measured effect of removing it
#
# Then rebuild the feature set without them.

### The lesson that matters most today

Look at the correlations of the two columns you just removed:

| Column | correlation with price | actual status |
|---|---:|---|
| `price_quote_price_per_night` | +1.00 | leak - correlation found it |
| `estimated_revenue_l365d` | +0.38 | leak - correlation would clear it |
| `price_quote_total_price` | **−0.05** | **leak - correlation says it is worthless** |

Look at the last row. `price_quote_total_price` has a correlation of about
**−0.05** with the target: statistically indistinguishable from no relationship at
all. On any screen - "drop features correlated above 0.9", or even "keep only
features correlated above 0.1" - it survives, or gets discarded as useless noise.
It is not borderline. It looks like nothing.

And removing it costs the gradient booster **0.12 R²** (0.957 → 0.834), because a
nonlinear model reconstructs `price = total_price ÷ nights` - and `nights` is
sitting right there in the quote dates.

> **Correlation with the target is a bad leak detector.** It only finds leaks that
> are *linear copies*. Leaks that are ratios, products, or conditional
> relationships are invisible to it - a near-zero correlation is not evidence of
> safety - and a flexible model will find them anyway.
>
> The reliable test is not statistical. It is: **what does this column mean, and
> would it exist when the prediction is needed?**

### Now build it properly

In [ ]:
# TODO: Build a ColumnTransformer + Pipeline that:
#   - imputes and scales the numeric features
#   - one-hot encodes the categoricals with min_frequency=20
#   - contains NO leaking columns
#   - has every transformer fitted INSIDE the pipeline
#
# Then score it with GroupKFold and confirm nothing is fitted outside.

### Why the pipeline is the structural fix

`cross_val_score` refits the *entire* pipeline on each training fold. The imputer's
medians, the scaler's means, the encoder's category list - all are learned from that
fold's training rows only, then applied to its validation rows.

Do it by hand and you will eventually fit one of them on everything. Not through
carelessness: through the ordinary accumulation of cells in a notebook over three
weeks. The pipeline makes the correct thing the easy thing.

### And is 0.83 honest?

We started at 0.999 and removed three leaks to reach roughly 0.83.

I am not going to tell you whether that number is trustworthy.

What I will tell you is that there is still something in this dataset that we have
not dealt with, and that it is not a leak in any of the four senses from §5. Write
**"is 0.83 real?"** at the top of your notebook. We will answer it in Session 6, and
you will find out then whether your instincts today were right.

## §7 - Common mistakes

| Mistake | Why it is tempting | What to do instead |
|---|---|---|
| Screening for leaks by correlation | It is one line of code | `total_price` correlates ≈0 and cost 0.12 R². Read what columns mean. |
| Celebrating a high score | Scores are the visible reward | A score above the field's ceiling is a bug report |
| Fixing one leak and stopping | The first explanation feels complete | We found three. Re-check after every fix. |
| `.fit()` on the full dataset "just to look" | Inspection feels passive | Fit inside the pipeline, inside the fold |
| Assuming a linear model is safer | It scored lower, so it leaked less | It leaked just as much and hid it better |
| Adding features until the score rises | It feels like progress | `km_centre` encoded a true belief and made things worse |
| Target encoding for high-cardinality columns | It reduces 69 columns to 1 and improves CV | It looks at the target. Fold-safe implementation or nothing. |

## §8 - Reflection

1. You applied the deployment test to `price_quote_total_price`. Now apply it to
   `number_of_reviews`, for Client B's use case. Is it legitimate? Argue both sides.
2. A colleague says "I always drop features correlated above 0.95 with the target,
   so I'm safe from leakage." Write the two-sentence reply.
3. Three leaks came from columns that Inside Airbnb *derived* rather than scraped.
   What does that suggest about using third-party derived fields as features?

## §9 - Knowledge check

1. Define data leakage in one sentence, without using the word "leak".
2. Why did the same leaking column give Ridge 0.855 and the booster 0.999?
3. `price_quote_total_price` correlates about −0.05 with the target - essentially
   zero. Explain how it can nevertheless leak.
4. Name the four leakage mechanisms and give one example of each.
5. What does putting a `SimpleImputer` inside a `Pipeline` prevent that calling
   `.fit_transform()` on the whole training set does not?

## Summary

- A feature should encode a stated belief. `km_centre` encoded a true belief and
  still made the model worse, because latitude and longitude already carried it.
- **Scaling is not universal hygiene.** It transformed KNN on wide-scale features
  (+0.24) and did nothing at all for trees (−0.0002). Match the treatment to the
  model.
- We removed three leaks and fell from **R² 0.999 → 0.834**:
  an identical column, a revenue ÷ occupancy ratio, and a total ÷ nights ratio.
- **A leak is not equally visible to every model.** The booster screamed at 0.999;
  Ridge quietly reported a plausible 0.855. The quiet case is the dangerous one.
- **Correlation is a bad leak detector.** The third leak correlated about −0.05
  with the target - indistinguishable from noise - and cost 0.12 R². Meaning beats
  statistics: ask what the column is and whether it exists at prediction time.
- Pipelines are the structural fix for contamination, because they make refitting
  per fold automatic rather than disciplined.

## Key takeaways

1. Distrust good news. A score above the field's ceiling is a bug report.
2. Fixing one leak does not mean you have fixed the leak.
3. "Would this value exist when the client needs the prediction?" - the cheapest
   and most reliable test you have.

## Further exploration

**Essential**
- Kapoor & Narayanan (2023), *Leakage and the reproducibility crisis in
  machine-learning-based science*, Patterns 4(9).
  https://arxiv.org/abs/2207.07048 - read it properly this week.
- scikit-learn user guide, *Common pitfalls and recommended practices*:
  https://scikit-learn.org/stable/common_pitfalls.html

**Recommended**
- Kaufman, Rosset & Perlich (2012), *Leakage in Data Mining: Formulation,
  Detection, and Avoidance*, ACM TKDD. The paper that named and systematised it.
- scikit-learn, `TargetEncoder` and its internal cross-fitting:
  https://scikit-learn.org/stable/modules/preprocessing.html#target-encoder

**Advanced**
- Micci-Barreca (2001), *A preprocessing scheme for high-cardinality categorical
  attributes*, SIGKDD Explorations - the origin of target encoding, and of the
  smoothing that makes it usable.

---

**Next session:** regression properly. Baselines first, then the model, then an
honest account of the error. Bring your audited pipeline - and your answer to
"is 0.83 real?"